# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Artasam/Machine-Learning/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook translates the ML-07 through ML-09 findings into a **concrete action playbook**
for FlyRank's content team: which pages to review first, why, what a human must check, and
when the recommendations go stale.

Following `writing-honest-claims/SKILL.md`:
- All claims use **observed / measured / directional / decision-support** language
- No causal claims without a controlled experiment
- Effect sizes with n, not drama
- Base rate printed next to every metric

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `writing-honest-claims` for this task.

In [1]:
%pip -q install duckdb huggingface_hub requests

In [2]:
import os, getpass

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token: ')

print('Token loaded:', 'YES ✅' if HF_TOKEN else 'NO ❌')

Token loaded: YES ✅


In [3]:
import duckdb
import pandas as pd
import numpy as np
import json, pathlib, csv

SEED = 42
np.random.seed(SEED)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL        = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Re-build SAME feature vector as ML-07, ML-08, ML-09 (identical SQL)
df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS prev_impressions,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks    ELSE 0 END)   AS prev_clicks,
        AVG(CASE WHEN report_date <= '2026-03-15' AND gsc_avg_position > 0
            THEN gsc_avg_position END)                                              AS prev_avg_position,
        SUM(CASE WHEN report_date <= '2026-03-15' AND gsc_impressions > 0
            THEN 1 ELSE 0 END)                                                      AS prev_days_active,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END)  AS imp_last15
    FROM {FACT_MARCH}
    GROUP BY content_hash_id, client_hash_id
    HAVING prev_impressions >= 50
""").df()

df['log_prev_impressions'] = np.log1p(df['prev_impressions'])
df['prev_ctr']             = df['prev_clicks'] / (df['prev_impressions'] + 1)
df['prev_avg_position']    = df['prev_avg_position'].fillna(50.0)
df['is_declining']         = (df['imp_last15'] < 0.8 * df['prev_impressions']).astype(int)

print(f"Pages: {len(df):,} | Base rate: {df['is_declining'].mean():.1%} | Clients: {df['client_hash_id'].nunique()}")
print("Data identical to ML-07 / ML-08 / ML-09 ✅")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 92,548 | Base rate: 28.6% | Clients: 40
Data identical to ML-07 / ML-08 / ML-09 ✅


---

## 1. Ranked actions + reason codes

**What:** The priority queue ranks every page by a composite score that combines search volume
(opportunity size), position tier (opportunity zone), and a CTR shield (pages already
converting are lower priority). This is the same rule-based scoring system validated in
ML-07 and ML-08.

**Why the rule, not the ML model?** In ML-08 and ML-09 we observed that the rule baseline
outperformed all three ML models (Logistic Regression, Random Forest, LightGBM) on unseen
clients. The rule achieved Precision@50 of 0.64 vs LightGBM's 0.44 on the grouped test set
(8 unseen clients, 22,531 pages, base rate 37.0%). Until richer features (content age,
keyword difficulty, text embeddings) are available, the rule is the production-ready tool.

**Each page in the queue gets a reason code** — a human-readable explanation of *why* it was
flagged, so the content team can decide whether to act without needing to understand the score.

In [4]:
# ======================================================================
# BUILD THE PRIORITY QUEUE (same rule as ML-07, now with reason codes)
# ======================================================================

POSITION_PENALTY = {
    'top_3': 0.3, 'page_1': 0.5, 'striking': 1.5,
    'page_3_5': 1.2, 'deep': 0.8, 'no_data': 0.5
}

def position_tier(pos):
    if pd.isna(pos) or pos == 0: return 'no_data'
    if pos <= 3:  return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

queue = df.copy()
queue['pos_tier'] = queue['prev_avg_position'].apply(position_tier)
queue['ctr_shield'] = 1 - np.minimum(queue['prev_ctr'] * 20, 0.5)
queue['action_score'] = (
    np.log1p(queue['prev_impressions'])
    * queue['pos_tier'].map(POSITION_PENALTY)
    * queue['ctr_shield']
)

# ── REASON CODES: human-readable explanation for each page
def build_reason(row):
    parts = []

    # Volume signal
    if row['prev_impressions'] >= 5000:
        parts.append(f"High-volume page ({int(row['prev_impressions']):,} impressions in 15 days)")
    elif row['prev_impressions'] >= 1000:
        parts.append(f"Moderate-volume page ({int(row['prev_impressions']):,} impressions)")
    else:
        parts.append(f"Lower-volume page ({int(row['prev_impressions']):,} impressions)")

    # Position signal
    tier = row['pos_tier']
    pos  = row['prev_avg_position']
    if tier == 'striking':
        parts.append(f"in striking distance (pos {pos:.1f}): small ranking gains yield large traffic gains")
    elif tier == 'page_3_5':
        parts.append(f"on page 3–5 (pos {pos:.1f}): needs significant improvement to reach page 1")
    elif tier == 'page_1':
        parts.append(f"on page 1 (pos {pos:.1f}): already visible but not top-3")
    elif tier == 'top_3':
        parts.append(f"in top 3 (pos {pos:.1f}): already dominant — low upside from refresh")
    elif tier == 'deep':
        parts.append(f"deeply ranked (pos {pos:.1f}): may need full rewrite rather than refresh")
    else:
        parts.append("no position data available")

    # CTR shield signal
    if row['prev_ctr'] > 0.025:
        parts.append(f"CTR is strong ({row['prev_ctr']:.3f}) — page is already converting well")
    elif row['prev_ctr'] < 0.005:
        parts.append(f"CTR is very low ({row['prev_ctr']:.4f}) — the page may not be compelling")

    return '; '.join(parts)

queue['reason_code'] = queue.apply(build_reason, axis=1)
queue['rank'] = queue['action_score'].rank(ascending=False, method='first').astype(int)
queue = queue.sort_values('rank')

print(f"Priority queue built: {len(queue):,} pages ranked")
print(f"Score range: {queue['action_score'].min():.3f} – {queue['action_score'].max():.3f}")
print()

Priority queue built: 92,548 pages ranked
Score range: 0.596 – 16.930



In [5]:
# ── DISPLAY TOP 20 with reason codes
print("TOP 20 CONTENT ACTION QUEUE")
print("=" * 100)
print(f"{'Rank':<6} {'Score':<8} {'Impressions':>12} {'Pos':>6} {'Tier':<12} {'CTR':>8} {'Reason Code'}")
print("-" * 100)

for _, row in queue.head(20).iterrows():
    print(f"{int(row['rank']):<6} {row['action_score']:<8.2f} "
          f"{int(row['prev_impressions']):>12,} {row['prev_avg_position']:>6.1f} "
          f"{row['pos_tier']:<12} {row['prev_ctr']:>8.4f}  "
          f"{row['reason_code'][:80]}")

print("=" * 100)
print()

# Verify precision on the queue (same as ML-07 full-pool evaluation)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['is_declining'].mean()
for k in [10, 20, 50, 100, 200]:
    p = precision_at_k(queue['action_score'].values, queue['is_declining'].values, k)
    lift = p / base_rate
    print(f"  P@{k:<4} = {p:.2f}  (lift {lift:.2f}x over {base_rate:.1%} base rate)")

TOP 20 CONTENT ACTION QUEUE
Rank   Score     Impressions    Pos Tier              CTR Reason Code
----------------------------------------------------------------------------------------------------
1      16.93         143,173   16.0 striking       0.0025  High-volume page (143,173 impressions in 15 days); in striking distance (pos 16.
2      16.60          70,169   18.3 striking       0.0004  High-volume page (70,169 impressions in 15 days); in striking distance (pos 18.3
3      16.30          72,940   18.2 striking       0.0015  High-volume page (72,940 impressions in 15 days); in striking distance (pos 18.2
4      15.48          32,902   11.0 striking       0.0004  High-volume page (32,902 impressions in 15 days); in striking distance (pos 11.0
5      15.16          25,299   19.6 striking       0.0002  High-volume page (25,299 impressions in 15 days); in striking distance (pos 19.6
6      15.15          25,896   15.2 striking       0.0003  High-volume page (25,896 impressions in 15

---

## 2. Intended use and limits

### Who uses this and for what

**Primary user:** FlyRank's content operations team — the people who decide which pages to
refresh, rewrite, or deprioritise each week.

**Intended use:** Open the priority queue CSV each Monday morning. Work the list from rank 1
downwards. For each page:
1. Read the **reason code** to understand why it was flagged
2. Check the page manually (see Section 3 below)
3. Decide: refresh, rewrite, or skip

**The queue is decision-support, not a decision.** It says *"these pages look worth reviewing
first, because…"* — it does NOT say *"refreshing these pages will increase traffic."*

### Where it stops being valid

| Limitation | Detail |
|---|---|
| **One month of data** | Built on March 2026 only. Seasonal patterns, algorithm updates, or industry shifts in other months are not captured. |
| **40 clients only** | The scoring was validated on 8 unseen clients (ML-08 grouped split). Performance on clients outside this portfolio is unknown. |
| **20% decline threshold is arbitrary** | A page dropping from 60 to 48 impressions crosses the threshold; a page dropping from 10,000 to 8,001 does not. The label does not distinguish business-critical declines from noise. |
| **No content-level signals** | The queue uses only traffic metrics (impressions, position, CTR, active days). It cannot detect content staleness, competitor moves, or topic relevance — a human must assess these. |
| **Observational only** | No controlled experiment was run. We observed an association between the score and subsequent decline, not a causal relationship. |

In [6]:
# ── QUANTIFY THE LIMITS with data

print("QUANTIFIED LIMITATIONS")
print("=" * 65)

# 1. Temporal coverage
print(f"\n1. TEMPORAL SCOPE")
print(f"   Feature window : Mar 01–15, 2026 (15 days)")
print(f"   Label window   : Mar 16–31, 2026 (16 days)")
print(f"   Coverage       : 1 month out of 17 available months")
print(f"   ⚠️  Findings are directional for this period only.")

# 2. Client coverage
n_clients = df['client_hash_id'].nunique()
pages_per_client = df.groupby('client_hash_id').size()
print(f"\n2. CLIENT COVERAGE")
print(f"   Total clients  : {n_clients}")
print(f"   Smallest client: {pages_per_client.min():,} pages")
print(f"   Largest client : {pages_per_client.max():,} pages")
print(f"   Median client  : {int(pages_per_client.median()):,} pages")
print(f"   ⚠️  Highly skewed: a few large clients dominate the queue.")

# 3. Label distribution by position tier
tier_stats = df.groupby(queue['pos_tier']).agg(
    n=('is_declining', 'count'),
    decline_rate=('is_declining', 'mean')
).sort_values('decline_rate', ascending=False)
print(f"\n3. DECLINE RATE BY POSITION TIER")
print(f"   {'Tier':<12} {'n':>8} {'Decline %':>10}")
print(f"   {'-'*12} {'-'*8} {'-'*10}")
for tier, row in tier_stats.iterrows():
    print(f"   {tier:<12} {row['n']:>8,} {row['decline_rate']:>9.1%}")

QUANTIFIED LIMITATIONS

1. TEMPORAL SCOPE
   Feature window : Mar 01–15, 2026 (15 days)
   Label window   : Mar 16–31, 2026 (16 days)
   Coverage       : 1 month out of 17 available months
   ⚠️  Findings are directional for this period only.

2. CLIENT COVERAGE
   Total clients  : 40
   Smallest client: 1 pages
   Largest client : 20,238 pages
   Median client  : 496 pages
   ⚠️  Highly skewed: a few large clients dominate the queue.

3. DECLINE RATE BY POSITION TIER
   Tier                n  Decline %
   ------------ -------- ----------
   page_3_5     17,880.0     32.4%
   page_1       42,867.0     29.5%
   deep          3,277.0     27.0%
   striking     18,731.0     26.1%
   top_3         9,793.0     23.6%


---

## 3. Human review + the no-go list

### What a person must check before acting

The queue **cannot** assess content quality, competitive landscape, or business context.
Before refreshing any flagged page, a human reviewer **must** check:

1. **Is the page seasonal?** A ski resort page declining in March is normal, not a problem.
   The queue has no seasonality adjustment — check the page's annual traffic pattern.
2. **Is the decline from an algorithm update?** If Google released a core update during
   Mar 16–31, the decline may be algorithmic rather than content-driven. A refresh may not help.
3. **Is the content still accurate?** A factually outdated page (wrong prices, old stats)
   needs a factual update, not an SEO refresh. The queue cannot detect factual staleness.
4. **Is the client still active?** Some clients may have paused their content program.
   Refreshing content for an inactive client wastes effort.
5. **Does the page have conversion value?** A page with 50,000 impressions but zero business
   relevance should not be prioritised over a page with 500 impressions that drives leads.

### The no-go list — what should NEVER be automated

| Never automate | Why |
|---|---|
| **Publishing a refresh without human review** | The queue ranks by traffic decline risk, not content quality. An automated refresh could damage a well-performing page. |
| **Deleting pages flagged as low-priority** | A low action score means "not in the current top priority batch" — not "this page has no value." |
| **Reporting the queue precision as "model accuracy"** | Precision@50 = 0.64 on unseen clients is a directional estimate, not a guarantee. Presenting it as accuracy would misrepresent the evidence. |
| **Using the queue for clients outside the training portfolio** | The rule was validated on 40 pseudonymised clients. Performance on new industries, languages, or content types is unverified. |

In [7]:
# ── HUMAN REVIEW DECISION TREE (printed for reference)

print("HUMAN REVIEW DECISION TREE")
print("=" * 65)
print("""
For each page in the queue (starting from rank 1):

  ┌─ Read the REASON CODE
  │
  ├─ CHECK 1: Is the page seasonal?
  │   YES → Skip (expected seasonal drop)
  │   NO  → Continue
  │
  ├─ CHECK 2: Was there a Google algorithm update?
  │   YES → Flag for monitoring, don't refresh yet
  │   NO  → Continue
  │
  ├─ CHECK 3: Is the content still factually correct?
  │   NO  → Prioritise factual update over SEO refresh
  │   YES → Continue
  │
  ├─ CHECK 4: Is the client still active?
  │   NO  → Skip (no point refreshing)
  │   YES → Continue
  │
  ├─ CHECK 5: Does the page have business value?
  │   NO  → Deprioritise (high traffic ≠ high value)
  │   YES → ✅ REFRESH THIS PAGE
  │
  └─ Log the decision (refresh / skip / monitor) for
     future model training data.
""")

# Count pages by position tier in the top 50
top50 = queue.head(50)
tier_counts = top50['pos_tier'].value_counts()
print("TOP 50 QUEUE — Position Tier Distribution:")
for tier, count in tier_counts.items():
    print(f"  {tier:<15} {count:>4} pages ({count/50:.0%})")

HUMAN REVIEW DECISION TREE

For each page in the queue (starting from rank 1):

  ┌─ Read the REASON CODE
  │
  ├─ CHECK 1: Is the page seasonal?
  │   YES → Skip (expected seasonal drop)
  │   NO  → Continue
  │
  ├─ CHECK 2: Was there a Google algorithm update?
  │   YES → Flag for monitoring, don't refresh yet
  │   NO  → Continue
  │
  ├─ CHECK 3: Is the content still factually correct?
  │   NO  → Prioritise factual update over SEO refresh
  │   YES → Continue
  │
  ├─ CHECK 4: Is the client still active?
  │   NO  → Skip (no point refreshing)
  │   YES → Continue
  │
  ├─ CHECK 5: Does the page have business value?
  │   NO  → Deprioritise (high traffic ≠ high value)
  │   YES → ✅ REFRESH THIS PAGE
  │
  └─ Log the decision (refresh / skip / monitor) for
     future model training data.

TOP 50 QUEUE — Position Tier Distribution:
  striking          49 pages (98%)
  page_3_5           1 pages (2%)


---

## 4. Monitoring / retrain triggers

The queue is a snapshot — it will go stale. These signals tell you when to re-run or re-build:

### Stale signals (re-run the queue with fresh data)
- **Monthly cadence:** The feature window is 15 days. After 30 days the traffic patterns
  may have shifted enough that the rankings are outdated. Re-run monthly at minimum.
- **Client portfolio change:** If FlyRank onboards or offboards a significant client,
  the base rate and position distributions shift. Re-run after major portfolio changes.

### Retrain triggers (re-build the scoring system)
- **Precision@50 drops below base rate** on a held-out evaluation set. If the queue is
  no longer better than random, the rule's assumptions have broken down.
- **New features become available:** Content age, keyword difficulty, or text embeddings
  would allow an ML model to potentially beat the rule. When these are available, revisit
  ML-08's model comparison.
- **Google algorithm update changes the game:** A major algorithm shift could invalidate
  the relationship between position/volume and decline risk.

In [8]:
# ── MONITORING DASHBOARD (baseline metrics to compare future runs against)

print("MONITORING BASELINE — compare future runs against these numbers")
print("=" * 70)
print()
print("Reference period: March 2026")
print(f"Total pages in queue     : {len(queue):,}")
print(f"Base rate (% declining)  : {df['is_declining'].mean():.1%}")
print(f"Clients in pool          : {df['client_hash_id'].nunique()}")
print()

print("BENCHMARK METRICS (full-pool, rule baseline):")
for k in [10, 20, 50, 100, 200]:
    p = precision_at_k(queue['action_score'].values, queue['is_declining'].values, k)
    print(f"  P@{k:<4} = {p:.2f}")

print()
print("ALERT THRESHOLDS:")
print(f"  ⚠️  Re-run queue if : data is > 30 days old")
print(f"  ⚠️  Re-run queue if : client portfolio changes by > 10%")
print(f"  🔴 Retrain rule if  : P@50 drops below base rate ({df['is_declining'].mean():.1%})")
print(f"  🔴 Retrain rule if  : new content-level features become available")
print(f"  🔴 Retrain rule if  : major Google algorithm update observed")

MONITORING BASELINE — compare future runs against these numbers

Reference period: March 2026
Total pages in queue     : 92,548
Base rate (% declining)  : 28.6%
Clients in pool          : 40

BENCHMARK METRICS (full-pool, rule baseline):
  P@10   = 0.60
  P@20   = 0.50
  P@50   = 0.42
  P@100  = 0.39
  P@200  = 0.39

ALERT THRESHOLDS:
  ⚠️  Re-run queue if : data is > 30 days old
  ⚠️  Re-run queue if : client portfolio changes by > 10%
  🔴 Retrain rule if  : P@50 drops below base rate (28.6%)
  🔴 Retrain rule if  : new content-level features become available
  🔴 Retrain rule if  : major Google algorithm update observed


---

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper
builds on these files.*

In [9]:
# ── EXPORT 1: Full priority queue CSV
out_dir = pathlib.Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

export_cols = [
    'rank', 'content_hash_id', 'client_hash_id',
    'action_score', 'pos_tier', 'prev_impressions',
    'prev_avg_position', 'prev_ctr', 'prev_days_active',
    'reason_code', 'is_declining'
]
queue_path = out_dir / 'action_queue.csv'
queue[export_cols].to_csv(queue_path, index=False)
print(f"✅ Full queue exported: {queue_path} ({len(queue):,} rows)")

✅ Full queue exported: work/outputs/action_queue.csv (92,548 rows)


In [10]:
# ── EXPORT 2: Playbook summary JSON (for the research paper)

base_rate = float(df['is_declining'].mean())

playbook_summary = {
    'notebook': 'w07_action_playbook.ipynb',
    'reference_period': 'March 2026',
    'seed': SEED,
    'data': {
        'total_pages': len(df),
        'total_clients': int(df['client_hash_id'].nunique()),
        'base_rate': round(base_rate, 4),
    },
    'recommended_system': 'Rule Baseline (ML-07)',
    'why_not_ml_model': (
        'LightGBM achieved P@50 = 0.44 on unseen clients vs '
        'Rule Baseline P@50 = 0.64 (ML-08 grouped split). '
        'ML model memorised client identity under random split '
        '(50-point gap at P@20, ML-09 audit). '
        'Rule is deployed until richer content-level features are available.'
    ),
    'precision_at_k': {
        f'P@{k}': round(float(precision_at_k(
            queue['action_score'].values, queue['is_declining'].values, k
        )), 4)
        for k in [10, 20, 50, 100, 200]
    },
    'limitations': [
        'Single month (Mar 2026) — no seasonality control',
        '40 clients — generalisation beyond this portfolio is unverified',
        '20% decline threshold is arbitrary',
        'No content-level signals — traffic metrics only',
        'Observational — no causal claims without a controlled experiment',
    ],
    'retrain_triggers': [
        f'P@50 drops below base rate ({base_rate:.1%})',
        'New content-level features available (content age, keyword difficulty)',
        'Major Google algorithm update',
        'Client portfolio changes by > 10%',
    ],
    'exports': [
        'work/outputs/action_queue.csv',
        'work/outputs/playbook_summary.json',
    ],
}

summary_path = out_dir / 'playbook_summary.json'
with open(summary_path, 'w') as f:
    json.dump(playbook_summary, f, indent=2)

print(f"✅ Playbook summary exported: {summary_path}")
print()
print(json.dumps(playbook_summary, indent=2))

✅ Playbook summary exported: work/outputs/playbook_summary.json

{
  "notebook": "w07_action_playbook.ipynb",
  "reference_period": "March 2026",
  "seed": 42,
  "data": {
    "total_pages": 92548,
    "total_clients": 40,
    "base_rate": 0.2864
  },
  "recommended_system": "Rule Baseline (ML-07)",
  "why_not_ml_model": "LightGBM achieved P@50 = 0.44 on unseen clients vs Rule Baseline P@50 = 0.64 (ML-08 grouped split). ML model memorised client identity under random split (50-point gap at P@20, ML-09 audit). Rule is deployed until richer content-level features are available.",
  "precision_at_k": {
    "P@10": 0.6,
    "P@20": 0.5,
    "P@50": 0.42,
    "P@100": 0.39,
    "P@200": 0.39
  },
  "limitations": [
    "Single month (Mar 2026) \u2014 no seasonality control",
    "40 clients \u2014 generalisation beyond this portfolio is unverified",
    "20% decline threshold is arbitrary",
    "No content-level signals \u2014 traffic metrics only",
    "Observational \u2014 no causal claim

---

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Section 1: Priority queue with reason codes, top-20 displayed, precision verified
- [x] Section 2: Intended use defined, 5 explicit limitations with data backing
- [x] Section 3: 5-step human review checklist + 4-item no-go list
- [x] Section 4: Stale signals + retrain triggers with concrete thresholds
- [x] Section 5: action_queue.csv + playbook_summary.json exported to work/outputs/
- [x] All numbers cross-checked against model_metrics.json and validation_audit_results.json
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.